# 07 · Silver · ERP (canal loja física)
Tipa os dados do ERP e marca os problemas de qualidade das vendas.
**Nenhuma linha é removida (D07):** cada problema tem a sua marcação, e o filtro acontece na gold (`WHERE valido`).

| Problema | Coluna | Regra |
|---|---|---|
| Venda sem produto | `sku_nulo` | SKU vazio |
| SKU fora do cadastro | `sku_sem_cadastro` | SKU preenchido que não existe em `erp_produtos` (esperado: 0) |
| Quantidade negativa | `quantidade_negativa` | Tratada como erro, não devolução: o ERP não tem devoluções (D17) |
| Linha repetida | `duplicada` | A 1ª ocorrência vale; as cópias idênticas recebem `true` |
| **Venda válida** | `valido` | Nenhum dos problemas acima |

Uma linha pode ter mais de um problema (por exemplo, a cópia de uma linha negativa), por isso cada um tem a sua coluna.

**Custo:** `custo_total = quantidade × custo` do `erp_produtos`. O ERP é o cadastro mestre de produtos (D12). É a base do CMV e da margem bruta.

**Equivalente no BigQuery** (mesma lógica; muda a sintaxe):
- `DECIMAL(12,2)` → `NUMERIC(12,2)`;
- `TIMESTAMP` → `DATETIME` (a hora do cupom é local, sem fuso; no BigQuery, `TIMESTAMP` seria interpretado como UTC);
- `COUNT_IF(x)` → `COUNTIF(x)`.

`ROW_NUMBER()`, `NULLIF`, `CAST` e as CTEs são iguais nos dois motores.

### 1. Cadastros

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.erp_lojas AS
SELECT loja_id, nome, cidade, uf, porte
FROM horizonte.bronze.erp_lojas;

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.erp_produtos AS
SELECT sku, nome, modelo, categoria, grupo,
       CAST(preco_tabela AS DECIMAL(12,2)) AS preco_tabela,
       CAST(custo AS DECIMAL(12,2))        AS custo
FROM horizonte.bronze.erp_produtos;

### 2. Vendas
Três etapas: tipar, marcar cada problema e combinar as marcações em `valido`.
As etapas ficam em CTEs separadas porque o BigQuery não deixa usar um alias criado no mesmo `SELECT`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.erp_vendas AS
WITH tipada AS (
  SELECT cupom_id,
         CAST(data_hora AS TIMESTAMP)          AS data_hora,
         loja_id,
         NULLIF(sku, '')                       AS sku,
         CAST(quantidade AS INT)               AS quantidade,
         CAST(preco_unitario AS DECIMAL(12,2)) AS preco_unitario,
         CAST(valor_total AS DECIMAL(12,2))    AS valor_total
  FROM horizonte.bronze.erp_vendas
),
marcada AS (
  SELECT t.*,
         p.categoria,
         p.grupo,
         CAST(t.quantidade * p.custo AS DECIMAL(12,2)) AS custo_total,
         t.sku IS NULL                                 AS sku_nulo,
         t.sku IS NOT NULL AND p.sku IS NULL           AS sku_sem_cadastro,
         t.quantidade < 0                              AS quantidade_negativa,
         ROW_NUMBER() OVER (
           PARTITION BY t.cupom_id, t.data_hora, t.loja_id, t.sku,
                        t.quantidade, t.preco_unitario, t.valor_total
           ORDER BY t.cupom_id) > 1                    AS duplicada
  FROM tipada t
  LEFT JOIN horizonte.silver.erp_produtos p ON p.sku = t.sku
)
SELECT cupom_id, data_hora, CAST(data_hora AS DATE) AS data, loja_id,
       sku, categoria, grupo,
       quantidade, preco_unitario, valor_total, custo_total,
       sku_nulo, sku_sem_cadastro, quantidade_negativa, duplicada,
       NOT (sku_nulo OR sku_sem_cadastro OR quantidade_negativa OR duplicada) AS valido
FROM marcada;

### 3. Estoque diário

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.erp_estoque_diario AS
SELECT CAST(data AS DATE)           AS data,
       loja_id,
       sku,
       CAST(estoque_inicial AS INT) AS estoque_inicial,
       CAST(entradas AS INT)        AS entradas,
       CAST(saidas AS INT)          AS saidas,
       CAST(estoque_final AS INT)   AS estoque_final
FROM horizonte.bronze.erp_estoque_diario;

### 4. Conferência
As marcações devem repetir o que a bronze encontrou (400 / 401 / 801). Receita e ticket médio das vendas válidas devem bater com o resumo do gerador (D17): R$ 30,8 mi e R$ 136,78.

In [0]:
SELECT COUNT(*)                     AS linhas,               -- 401.335
       COUNT_IF(sku_nulo)           AS sku_nulo,             -- 400
       COUNT_IF(sku_sem_cadastro)   AS sku_sem_cadastro,     -- 0
       COUNT_IF(quantidade_negativa) AS quantidade_negativa, -- 401
       COUNT_IF(duplicada)          AS duplicada,            -- 801
       COUNT_IF(valido)             AS validas,              -- um pouco mais que 399.733 (linhas com mais de um problema)
       SUM(CASE WHEN valido THEN valor_total END) AS receita_valida   -- cerca de R$ 30,8 mi
FROM horizonte.silver.erp_vendas;

In [0]:
SELECT ROUND(AVG(total_cupom), 2) AS ticket_medio   -- 136,78
FROM (
  SELECT cupom_id, SUM(valor_total) AS total_cupom
  FROM horizonte.silver.erp_vendas
  WHERE valido
  GROUP BY cupom_id
);

In [0]:
SELECT 'erp_lojas' AS tabela, COUNT(*) AS linhas FROM horizonte.silver.erp_lojas      -- 12
UNION ALL SELECT 'erp_produtos', COUNT(*) FROM horizonte.silver.erp_produtos             -- 200
UNION ALL SELECT 'erp_estoque_diario', COUNT(*) FROM horizonte.silver.erp_estoque_diario; -- 1.258.331